# Q8: Eigen Values and Eigen Vectors
**Topic:** Coding-question | **Difficulty:** Medium | **Time:** 20 min
**Sheet:** Grind75ML

---

## Question
Implement computation of Eigenvalues and Eigenvectors from scratch using the Power Iteration method.

## Theory

For a square matrix $A$, an eigenvector $v$ and eigenvalue $\lambda$ satisfy:

$$Av = \lambda v$$

**Interpretation:** An eigenvector is a direction that only gets scaled (not rotated) when the matrix is applied. The eigenvalue is the scaling factor.

**Power Iteration** finds the dominant eigenvalue/vector by repeatedly multiplying a random vector by $A$ and normalizing:

1. Start with random vector $v_0$
2. $v_{k+1} = \frac{A v_k}{\|A v_k\|}$
3. $\lambda = \frac{v^T A v}{v^T v}$ (Rayleigh quotient)

**Applications:** PCA, Google PageRank, stability analysis, quantum mechanics.

In [ ]:
import numpy as np


def power_iteration(A: np.ndarray, num_iters: int = 100, tol: float = 1e-10) -> tuple:
    """Find the dominant eigenvalue and eigenvector using Power Iteration.
    
    Args:
        A: Square matrix (n x n).
        num_iters: Maximum iterations.
        tol: Convergence tolerance.
    
    Returns:
        Tuple of (dominant_eigenvalue, dominant_eigenvector).
    """
    n = A.shape[0]
    # Start with a random vector
    v = np.random.randn(n)
    v = v / np.linalg.norm(v)
    
    for _ in range(num_iters):
        # Multiply by A
        Av = A @ v
        # Normalize
        v_new = Av / np.linalg.norm(Av)
        # Check convergence
        if np.abs(np.abs(v_new @ v) - 1.0) < tol:
            v = v_new
            break
        v = v_new
    
    # Rayleigh quotient for eigenvalue
    eigenvalue = (v @ A @ v) / (v @ v)
    return eigenvalue, v


def find_all_eigenvalues(A: np.ndarray) -> tuple:
    """Find all eigenvalues/vectors using deflation + power iteration.
    
    After finding each dominant eigenvalue, we deflate the matrix:
    A_new = A - lambda * v * v^T
    """
    n = A.shape[0]
    eigenvalues = []
    eigenvectors = []
    A_current = A.copy().astype(float)
    
    for _ in range(n):
        eigval, eigvec = power_iteration(A_current)
        eigenvalues.append(eigval)
        eigenvectors.append(eigvec)
        # Deflation: remove contribution of found eigenvalue
        A_current = A_current - eigval * np.outer(eigvec, eigvec)
    
    return np.array(eigenvalues), np.array(eigenvectors).T

In [ ]:
# --- Example ---
np.random.seed(42)
A = np.array([[4, 1, 2],
              [1, 3, 1],
              [2, 1, 5]], dtype=float)

# Our implementation
eigenvalues, eigenvectors = find_all_eigenvalues(A)
print("=== Our Implementation ===")
print(f"Eigenvalues:  {eigenvalues}")
print(f"Eigenvectors:\n{eigenvectors}")

# NumPy's built-in (for verification)
np_vals, np_vecs = np.linalg.eigh(A)  # eigh for symmetric matrices
print("\n=== NumPy Verification ===")
print(f"Eigenvalues:  {np.sort(np_vals)[::-1]}")
print(f"Eigenvectors:\n{np_vecs}")

# Verify: Av = lambda * v
print("\n=== Verification: Av = λv ===")
for i in range(len(eigenvalues)):
    Av = A @ eigenvectors[:, i]
    lv = eigenvalues[i] * eigenvectors[:, i]
    print(f"λ={eigenvalues[i]:.4f}: ||Av - λv|| = {np.linalg.norm(Av - lv):.2e}")

## Key Takeaways
- **Power Iteration** converges to the dominant (largest magnitude) eigenvalue — simple but effective
- **Deflation** removes found eigenvalues to discover subsequent ones
- In practice, use `np.linalg.eig()` (general) or `np.linalg.eigh()` (symmetric) — they use LAPACK routines
- PCA is fundamentally eigendecomposition of the covariance matrix